# Spatial Equilibrium Solver

> Iterative algorithm for finding the equilibrium distribution of workers across cities


In [1]:
# | default_exp equilibrium

In [3]:
# | export
import numpy as np
from scipy.special import logsumexp as scipy_logsumexp

from src.parameters import CityParameters, EquilibriumResult, ModelParameters

## The Spatial Equilibrium Concept

In spatial equilibrium, workers are **indifferent between locations**. If one city were strictly better, workers would migrate there, changing wages and rents until the advantage disappears.

### Two Equilibrium Concepts

**1. Perfect Mobility (Classic Rosen-Roback)**

Workers are perfectly mobile, so utilities **exactly equalize**:

$$V_1 = V_2 = V_3 = \ldots = \bar{V}$$

In this case, amenities are **calibrated** to rationalize observed populations given wages and rents.

**2. Imperfect Mobility (Fréchet/Probabilistic)**

Workers face moving costs or have heterogeneous location preferences. Population shares follow:

$$\text{share}_i = \frac{V_i^\theta}{\sum_j V_j^\theta}$$

where $\theta$ is the **mobility parameter**:

- Higher $\theta$ → more mobile → utilities converge
- Lower $\theta$ → less mobile → larger utility differences
- $\theta \to \infty$ → perfect mobility (classic case)

This model allows realistic utility spreads (e.g., 3-15%) while maintaining spatial equilibrium logic.

## The Algorithm

We solve for equilibrium iteratively:

1. **Start** with equal population distribution
2. **Calculate** wages and rents from population (using production and housing market equilibrium)
3. **Calculate** utility in each city
4. **Update** population shares using $\text{share}_i \propto V_i^\theta$
5. **Repeat** until convergence

### Why This Works

The algorithm mimics **worker migration**:

- If City A has higher utility than City B, workers flow from B to A
- This raises rents in A and lowers rents in B
- It also changes wages (through agglomeration and diminishing returns)
- Process continues until no one wants to move

## Numerical Stability: Log-Space Calculations

Working directly with populations and utilities can cause numerical problems:

- Populations can be millions → overflow when raised to powers
- Utilities raised to $\theta$ can be very small → underflow
- Share calculations involve ratios of exponentials → numerical instability

**Solution**: Do all calculations in log-space!

### The Log-Sum-Exp Trick

To compute $\log(\sum_i e^{x_i})$ without overflow:

$$\log\left(\sum_i e^{x_i}\right) = c + \log\left(\sum_i e^{x_i - c}\right)$$

where $c = \max_i x_i$. This shifts all terms so the maximum is 0, preventing overflow.


## The Equilibrium Solver


In [ ]:
# | export
class EquilibriumSolver:
    """Iterative solver for spatial equilibrium using the Fréchet/probabilistic model."""

    def __init__(
        self,
        params: ModelParameters,  # Model parameters (elasticities, shares)
        cities: list[CityParameters],  # List of city-specific parameters
    ):
        """Initialize the solver."""
        self.params = params
        self.cities = cities
        self.n_cities = len(cities)

        # Pre-compute log of city-specific parameters for efficiency
        self._log_base_tfp = np.array([np.log(c.base_tfp) for c in cities])
        self._log_amenity = np.array([np.log(c.amenity) for c in cities])
        self._supply_elasticity = np.array([c.supply_elasticity for c in cities])
        self._log_supply_shifter = np.array([np.log(c.supply_shifter) for c in cities])

    def log_rent_from_log_population(
        self,
        log_pop: np.ndarray,  # Log population in each city
    ) -> np.ndarray:
        """Calculate log rents from log population using housing market equilibrium."""
        return (log_pop - self._log_supply_shifter) / self._supply_elasticity

    def log_wage_from_log_population(
        self,
        log_pop: np.ndarray,  # Log population in each city
    ) -> np.ndarray:
        """Calculate log wages from log population using production function."""
        p = self.params
        return np.log(p.alpha) + self._log_base_tfp + (p.alpha + p.eta - 1) * log_pop

    def log_utility(
        self,
        log_wages: np.ndarray,  # Log wages in each city
        log_rents: np.ndarray,  # Log rents in each city
    ) -> np.ndarray:
        """Calculate log utility in each city."""
        return log_wages - self.params.beta * log_rents + self._log_amenity

    def log_output(
        self,
        log_pop: np.ndarray,  # Log population in each city
    ) -> np.ndarray:
        """Calculate log output in each city."""
        return self._log_base_tfp + (self.params.alpha + self.params.eta) * log_pop

    def _logsumexp(
        self,
        x: np.ndarray,  # Array of values
    ) -> float:
        """Compute log(sum(exp(x))) without overflow using scipy's implementation."""
        return float(scipy_logsumexp(x))

    def _log_softmax(
        self,
        x: np.ndarray,  # Array of values
    ) -> np.ndarray:
        """Compute log(softmax(x)) in a numerically stable way."""
        return x - scipy_logsumexp(x)

    def solve(
        self,
        total_population: float,  # Total population to distribute across cities
        tol: float = 1e-8,  # Convergence tolerance (max change in population shares)
        max_iter: int = 2000,  # Maximum iterations before giving up
        damping: float = 0.3,  # Damping factor for updates (0-1, lower = more stable)
        verbose: bool = False,  # If True, print iteration progress
    ) -> EquilibriumResult:
        """Solve for spatial equilibrium where worker shares follow the Fréchet distribution."""
        # Initial guess: equal distribution in log space
        log_total_pop = np.log(total_population)
        log_pop = np.full(self.n_cities, log_total_pop - np.log(self.n_cities))

        # Adaptive damping for high theta (more mobile workers need gentler updates)
        # Use very strong damping for high theta to prevent oscillation
        effective_damping = min(damping, 0.5 / (1.0 + self.params.theta / 5.0))

        converged = False
        iteration = 0
        for iteration in range(max_iter):
            # Calculate wages and rents in log space
            log_rents = self.log_rent_from_log_population(log_pop)
            log_wages = self.log_wage_from_log_population(log_pop)
            log_utilities = self.log_utility(log_wages, log_rents)

            # Workers move toward high-utility cities
            # share_i proportional to utility_i^theta
            # Use log_softmax for numerical stability
            log_shares = self._log_softmax(self.params.theta * log_utilities)

            # Check for numerical issues
            if not np.all(np.isfinite(log_shares)):
                # Fall back to equal shares if numerical issues
                log_shares = np.full(self.n_cities, -np.log(self.n_cities))

            # New population in log space
            new_log_pop = log_shares + log_total_pop

            # Damped update for stability
            new_log_pop = effective_damping * new_log_pop + (1 - effective_damping) * log_pop

            # Check convergence on population shares
            old_shares = np.exp(self._log_softmax(log_pop))
            new_shares = np.exp(self._log_softmax(new_log_pop))
            diff = np.abs(new_shares - old_shares).max()

            if verbose and iteration % 100 == 0:
                utilities = np.exp(log_utilities)
                util_spread = (utilities.max() / utilities.min() - 1) * 100
                print(f"Iteration {iteration}: share change = {diff:.2e}, utility spread = {util_spread:.2f}%")

            log_pop = new_log_pop

            if diff < tol:
                converged = True
                break

        # Final calculations (convert back from log space)
        population = np.exp(log_pop)
        # Renormalize to ensure exact total
        population = population * (total_population / population.sum())

        log_pop = np.log(population)
        log_rents = self.log_rent_from_log_population(log_pop)
        log_wages = self.log_wage_from_log_population(log_pop)
        log_utilities = self.log_utility(log_wages, log_rents)
        log_outputs = self.log_output(log_pop)

        rents = np.exp(log_rents)
        wages = np.exp(log_wages)
        utilities = np.exp(log_utilities)
        outputs = np.exp(log_outputs)

        return EquilibriumResult(
            city_names=[c.name for c in self.cities],
            population=population.tolist(),
            wages=wages.tolist(),
            rents=rents.tolist(),
            utilities=utilities.tolist(),
            outputs=outputs.tolist(),
            total_gdp=float(outputs.sum()),
            converged=converged,
            iterations=iteration + 1,
        )

## Helper Function for Counterfactual Analysis


In [ ]:
# | export
def calculate_gdp_change(
    baseline: EquilibriumResult,  # Baseline equilibrium result
    counterfactual: EquilibriumResult,  # Counterfactual equilibrium result
) -> dict[str, float]:
    """Calculate GDP change between two equilibria."""
    gdp_change = counterfactual.total_gdp - baseline.total_gdp
    gdp_change_pct = (gdp_change / baseline.total_gdp) * 100

    return {
        "baseline_gdp": baseline.total_gdp,
        "counterfactual_gdp": counterfactual.total_gdp,
        "gdp_change": gdp_change,
        "gdp_change_pct": gdp_change_pct,
    }

## Tests


In [6]:
# | hide
# Test the solver with simple three-city example
params = ModelParameters(alpha=0.65, eta=0.04, beta=0.33, theta=10.0)

cities = [
    CityParameters(name="High", base_tfp=120, amenity=1.0, supply_elasticity=1.5, supply_shifter=1000),
    CityParameters(name="Medium", base_tfp=100, amenity=1.0, supply_elasticity=2.0, supply_shifter=1000),
    CityParameters(name="Low", base_tfp=80, amenity=1.0, supply_elasticity=3.0, supply_shifter=1000),
]

solver = EquilibriumSolver(params, cities)
result = solver.solve(total_population=3_000_000)

# Test that solution has correct properties
assert result.converged
assert len(result.population) == 3
assert len(result.wages) == 3
assert len(result.rents) == 3
assert len(result.utilities) == 3

# Test that population sums to total
assert np.isclose(sum(result.population), 3_000_000)

# Test that all values are positive
assert all(p > 0 for p in result.population)
assert all(w > 0 for w in result.wages)
assert all(r > 0 for r in result.rents)
assert all(u > 0 for u in result.utilities)

# Test GDP change calculation
# Make second scenario with higher elasticity in first city
cities2 = [
    CityParameters(name="High", base_tfp=120, amenity=1.0, supply_elasticity=3.0, supply_shifter=1000),
    CityParameters(name="Medium", base_tfp=100, amenity=1.0, supply_elasticity=2.0, supply_shifter=1000),
    CityParameters(name="Low", base_tfp=80, amenity=1.0, supply_elasticity=3.0, supply_shifter=1000),
]
solver2 = EquilibriumSolver(params, cities2)
result2 = solver2.solve(total_population=3_000_000)

gdp_change = calculate_gdp_change(result, result2)
assert "baseline_gdp" in gdp_change
assert "counterfactual_gdp" in gdp_change
assert "gdp_change" in gdp_change
assert "gdp_change_pct" in gdp_change

# Increasing elasticity in high-productivity city should increase GDP
assert gdp_change["gdp_change"] > 0

## Example Usage


In [7]:
# Solve equilibrium for three cities
params = ModelParameters.STANDARD

cities = [
    CityParameters(name="High Productivity", base_tfp=120, supply_elasticity=1.5),
    CityParameters(name="Medium", base_tfp=100, supply_elasticity=2.0),
    CityParameters(name="Low Productivity", base_tfp=80, supply_elasticity=3.0),
]

solver = EquilibriumSolver(params, cities)
result = solver.solve(total_population=3_000_000, verbose=False)

print(result.summary())

Spatial Equilibrium Results
City                        Population       Wage       Rent    Utility
----------------------------------------------------------------------
High Productivity              713,304       1.20      79.83     0.2817
Medium                         973,846       0.90      31.21     0.2907
Low Productivity             1,312,850       0.66      10.95     0.2995
----------------------------------------------------------------------
Total GDP: 3,999,865
Utility spread: 6.3% (smaller = closer to classic R-R equilibrium)
Converged: True (64 iterations)


## Key Insights

1. **Spatial equilibrium balances three forces**:

   - Agglomeration (pulls workers to productive cities)
   - Housing supply constraints (push workers away via high rents)
   - Worker mobility (how responsive to utility differences)

2. **Log-space calculations are essential**: Without them, the solver would fail due to overflow/underflow

3. **Damping prevents oscillation**: With high $\theta$, workers are very responsive, causing the algorithm to overshoot. Damping stabilizes convergence.

4. **Utility spread reveals mobility**:

   - 1-3% spread → high mobility (θ > 20)
   - 5-10% spread → moderate mobility (θ ≈ 5-10)
   - 10-20% spread → low mobility (θ ≈ 2-4)

5. **The algorithm is robust**: It handles any number of cities, various parameter values, and finds equilibrium reliably


In [8]:
# | hide
import nbdev

nbdev.nbdev_export()